# Day 1 · SONIC 해부하기 — 모델과 데이터의 '계약' 읽기
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kehdgus9434/sonic-vibe-coding-lab/blob/main/notebooks/day1_anatomy.ipynb)

**오늘의 결과물**: 공식 걷기 샘플 → *내가 만든* Encoder 입력(1762) → 64차원 토큰 시계열

| 시간 | 내용 |
|---|---|
| 0:00–0:15 | 환경 준비 · 목적지 미리보기 · 도구 연결 |
| 0:15–0:30 | Mission 1-1 ONNX 입출력 해부 |
| 0:30–0:50 | Mission 1-2 Encoder 1762차원 블록 표 |
| 0:50–1:10 | Mission 1-3 쿼터니언 순서 탐정 |
| 1:10–1:25 | Mission 1-4 30 Hz ↔ 50 Hz 시간 정렬 |
| 1:25–1:50 | Mission 1-5 Mode-2 Encoder 입력 조립 → 토큰 |
| 1:50–2:00 | 정리 · (빨리 끝나면 🚀 심화) |


**AI 사용 규칙 (이 과정은 AI 에이전트 사용을 전제로 합니다)**
- 코드는 어떤 도구로 써도 됩니다: Colab의 Gemini, 웹 채팅(ChatGPT·Claude·Gemini), 웹 코딩 에이전트(Claude Code on the web, Codex) — 설치 없이 브라우저만으로 (`docs/00_setup.md`).
- 대신 **평가는 코드가 아니라 당신의 판단**입니다: 🔮 예측(실행 전)과 🧾 검증 기록(체크 후)은 **본인이 직접** 씁니다.
- `sonic_lab/reference.py`·`check.py`는 정답입니다. 에이전트에게도 읽지 말라고 지시하세요(`AGENTS.md`에 규칙이 있음).

⚠️ 이 실습의 어떤 출력도 실물 로봇에 보내지 않습니다.

In [ ]:
#@title 0. 실습 환경 준비 — 처음 한 번 실행 (약 1–2분)
REPO = "https://github.com/kehdgus9434/sonic-vibe-coding-lab"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
# 웹 코딩 에이전트 사용자: PR 댓글의 "Colab에서 내 브랜치로 열기" 링크를 쓰면 이 두 값이 자동으로 채워집니다
import os, sys, subprocess
from pathlib import Path
if Path("/content").exists():                              # Google Colab
    LAB_DIR = Path("/content/sonic-vibe-coding-lab")
    if not LAB_DIR.exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, REPO, str(LAB_DIR)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(LAB_DIR / "requirements-colab.txt")], check=True)
else:                                                      # 로컬 Jupyter / VS Code (저장소 안에서 실행)
    LAB_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(LAB_DIR)
sys.path.insert(0, str(LAB_DIR))
import numpy as np
import sonic_lab
from sonic_lab import check, reference as ref, render, analysis
lab = sonic_lab.start()

In [ ]:
#@title 🔄 (웹 에이전트 사용자) 에이전트가 Fork에 push한 최신 코드 받기 — 필요할 때만
r = subprocess.run(["git", "-C", str(LAB_DIR), "pull", "--ff-only", "origin", BRANCH], capture_output=True, text=True)
print(r.stdout or r.stderr)
for name in [m for m in sys.modules if m.startswith("my_work")]:
    del sys.modules[name]                                  # 다음 import에서 새 코드를 읽도록
print("완료 — my_work 코드를 쓰는 ✍️ 셀을 다시 실행하세요")

## 목적지 미리보기 (5분)
참조 구현으로 3일 뒤 도달할 결과를 먼저 봅니다: **사람 걷기 데이터 → SONIC → MuJoCo G1**.

> 🔮 **예측 — 아래 셀을 실행하기 전에 직접 적으세요 (AI에게 맡기지 않기)**
> 사람(SMPL) 걷기 데이터를 SONIC에 넣으면, 처음 5초(250프레임) 동안 G1은 넘어지지 않을까요? 넘어진다면 몇 초쯤?
>
> 내 예측: ______

In [ ]:
summary, arrays = ref.run_physics(lab.sim, lab.policy, lab.motion, mode=2, max_frames=250)
print({k: summary[k] for k in ["simulated_frames", "requested_frames", "completed", "stop_reason", "runtime_s"]})
render.show(render.skeleton_video(lab, arrays, "outputs/day1_preview.mp4", title="reference pipeline", seconds=5))

---
## Mission 1-1 · ONNX 입출력 해부 (15분)
> 🧭 아래 설명 + `docs/01_contract.md` + 함수 시그니처를 AI에게 주고 ✍️ 셀을 채운 뒤 ✅ 셀로 확인합니다.

- **목표**: `model_encoder.onnx`, `model_decoder.onnx`의 입력·출력 이름과 shape를 확인한다.
- **주어진 것**: `lab.assets.model_dir` (ONNX 2개 + `observation_config.yaml`)
- **완료 기준**: ✅ 셀 통과

In [ ]:
# ✍️ Mission 1-1
def inspect_onnx(model_dir):
    """반환: (encoder 입력 길이, encoder 출력 길이, decoder 입력 길이, decoder 출력 길이)"""
    raise NotImplementedError

# from my_work.day1 import *    # 에이전트형 도구로 my_work/day1.py에 작성했다면 주석 해제

In [ ]:
# ✅ 확인
check.onnx_io(*inspect_onnx(lab.assets.model_dir))

> 🧾 **검증 기록 — 직접 적으세요**
> `observation_config.yaml` 맨 위 주석의 *Total dimension* 값과 실제 Decoder 입력 길이가 같나요? 어느 쪽을 믿어야 하고, 왜인가요?
>
> 근거: ______

---
## Mission 1-2 · Encoder 1762차원 블록 표 (20분)
> 🧭 아래 설명 + `docs/01_contract.md` + 함수 시그니처를 AI에게 주고 ✍️ 셀을 채운 뒤 ✅ 셀로 확인합니다.

- **목표**: `observation_config.yaml`을 읽어 Encoder 관측 14개 블록의 `(start, stop)` 표를 코드로 만든다.
- **함정**: YAML에는 **길이가 적혀 있지 않습니다.** 이름(`10frame`, `step5`, `lowerbody`, `vr_3point` …)과 계약서 3절로 길이를 정하고, 합이 ONNX 입력 길이와 같은지 `assert`로 확인하게 하세요.
- **완료 기준**: ✅ 셀 통과

In [ ]:
# ✍️ Mission 1-2
def build_encoder_slices(yaml_path):
    """반환: {관측 이름: (start, stop)} — YAML의 enabled 순서"""
    raise NotImplementedError

# from my_work.day1 import *

In [ ]:
# ✅ 확인
slices = build_encoder_slices(lab.assets.model_dir / "observation_config.yaml")
check.encoder_slices(lab, slices)

> 🧾 **검증 기록 — 직접 적으세요**
> AI가 블록 길이를 정할 때 계약서에 **없는** 규칙을 지어낸 적이 있었나요? `vr_3point_local_orn_target`이 왜 12인지 직접 설명할 수 있나요?
>
> 근거: ______

---
## Mission 1-3 · 쿼터니언 순서 탐정 (20분)
> 🧭 아래 설명 + `docs/01_contract.md` + 함수 시그니처를 AI에게 주고 ✍️ 셀을 채운 뒤 ✅ 셀로 확인합니다.

공식 G1 샘플에는 루트 회전이 **두 가지 표현**으로 들어 있습니다: `pose_aa[:, 0]`(축-각 벡터)과 `root_rot`(쿼터니언).
실제 연구 과정(06번 노트북)에서 `root_rot`를 잘못된 성분 순서로 읽어 큰 회전 오류가 난 적이 있습니다.

- **목표**: 두 표현을 비교해 `root_rot`가 `xyzw`인지 `wxyz`인지 **데이터로** 판정하는 함수를 만든다.
- **요청 팁**: "어느 쪽일 것 같아?"가 아니라 "두 가정 각각의 최대 각도 오차를 계산해서 비교하는 코드"를 요청하세요.

In [ ]:
# 데이터 불러오기 (제공)
from sonic_lab.reference import load_published_pickle
from sonic_lab.assets import SAMPLE
robot = load_published_pickle(lab.assets.sample_root / "robot_filtered/210531" / SAMPLE)
human = load_published_pickle(lab.assets.sample_root / "smpl_filtered" / SAMPLE)
print("G1  :", {k: np.shape(v) for k, v in robot.items()})
print("SMPL:", {k: np.shape(v) for k, v in human.items()})

> 🔮 **예측 — 아래 셀을 실행하기 전에 직접 적으세요 (AI에게 맡기지 않기)**
> 성분 순서를 잘못 읽으면 최대 각도 오차는 몇 도쯤 될까요? (5°? 45°? 180°?)
>
> 내 예측: ______

In [ ]:
# ✍️ Mission 1-3
def detect_root_quat_order(robot_record):
    """반환: 'xyzw' 또는 'wxyz' (판정 근거가 되는 두 오차도 출력할 것)"""
    raise NotImplementedError

# from my_work.day1 import *

In [ ]:
# ✅ 확인
check.root_quat_order(detect_root_quat_order(robot))

---
## Mission 1-4 · 30 Hz ↔ 50 Hz 시간 정렬 (15분)
> 🧭 아래 설명 + `docs/01_contract.md` + 함수 시그니처를 AI에게 주고 ✍️ 셀을 채운 뒤 ✅ 셀로 확인합니다.

- G1 참조는 30 fps, 사람(SMPL)은 50 fps이고 길이도 조금 다릅니다. 정책은 50 Hz로 돕니다.
- **목표**: 두 데이터 중 **짧은 쪽 길이**에 맞춘 50 Hz 시간축과, 그 시간에서 선형 보간한 G1 관절각 `(T, 29)`를 만든다.

In [ ]:
# ✍️ Mission 1-4
def resample_g1(robot_record, human_record, hz=50):
    """반환: (times (T,), g1_q50 (T, 29))"""
    raise NotImplementedError

# from my_work.day1 import *

In [ ]:
# ✅ 확인
times, g1_q50 = resample_g1(robot, human)
check.resampled_motion(lab, times, g1_q50)

---
## Mission 1-5 · Mode-2 Encoder 입력 조립 → 토큰 (25분)
> 🧭 아래 설명 + `docs/01_contract.md` + 함수 시그니처를 AI에게 주고 ✍️ 셀을 채운 뒤 ✅ 셀로 확인합니다.

`lab.motion`은 이미 50 Hz로 정렬된 공식 샘플입니다(앞 미션을 못 끝냈어도 진행 가능).

| 필드 | shape | 의미 |
|---|---|---|
| `lab.motion.local24` | (T, 24, 3) | 루트 기준 SMPL 관절 좌표 |
| `lab.motion.human_root_xyzw` | (T, 4) | 인체 루트 회전 (scipy xyzw) |
| `lab.motion.wrists_il` | (T, 6) | G1 손목 6관절 (IsaacLab 23–28) |
| `lab.motion.times` | (T,) | 50 Hz 시간축 |

- **목표**: 계약서 3절대로 `frame` 시점의 1762차원 Mode-2 Encoder 입력을 만든다. `robot_xyzw`는 현재 로봇 기저 회전(xyzw), `alignment`는 scipy `Rotation`(헤딩 정렬 A)입니다.

In [ ]:
# ✍️ Mission 1-5
from scipy.spatial.transform import Rotation as R

def build_encoder_obs_mode2(motion, frame, robot_xyzw, alignment):
    """반환: (1762,) float32"""
    raise NotImplementedError

# from my_work.day1 import *

In [ ]:
# ✅ 확인
check.encoder_obs(lab, build_encoder_obs_mode2)

> 🧾 **검증 기록 — 직접 적으세요**
> 6D 방향 블록에서 회전행렬의 '열'과 '행'을 바꿔 넣어도 체크를 통과할까요? 왜 그렇게 생각하는지, 혹은 직접 바꿔 본 결과를 적으세요.
>
> 근거: ______

> 🔮 **예측 — 아래 셀을 실행하기 전에 직접 적으세요 (AI에게 맡기지 않기)**
> 걷기 샘플의 토큰 64차원 중 일부는 걸음 주기(약 1초)를 따라 주기적으로 움직일까요?
>
> 내 예측: ______

In [ ]:
# 내 Encoder 입력으로 만든 토큰 시계열 (처음 10초, 로봇이 참조와 같은 방향이라고 가정)
import matplotlib.pyplot as plt
from scipy.spatial.transform import Rotation as R
A = R.identity()
tokens = np.array([lab.policy.encode_obs(build_encoder_obs_mode2(lab.motion, t, lab.motion.human_root_xyzw[t], A))
                   for t in range(0, 500, 2)])
plt.figure(figsize=(11, 3))
plt.plot(lab.motion.times[0:500:2], tokens[:, :8]); plt.xlabel("time (s)"); plt.title("SONIC token dims 0–7 (walk sample)")
plt.show()

---
## 정리 (10분)
- 오늘 🔮 예측이 틀렸던 것과, AI가 틀렸던 순간을 무엇으로 알아챘는지 옆 사람과 이야기해 보세요.
- 빨리 끝났다면 아래 🚀 심화 미션으로 넘어갑니다.

---
## 🚀 심화 미션 (빨리 끝난 경우, 정답 비교가 있는 것과 없는 것)

**A. Mode-0 Encoder 입력 (체크 있음)** — G1 참조 관절각으로 Encoder 입력을 만듭니다(계약서 3절의 `step5` 블록).
`lab.motion.g1_q`(MuJoCo 순서 관절각), `lab.motion.g1_dq`(관절 속도, 제공), `lab.motion.g1_root_xyzw`(G1 루트 회전)를 씁니다.

**B. 좌우 반전 샘플의 토큰 (정답 없음)** — `lab.load_motion(mirrored=True)`로 반전 샘플을 불러 원본과 토큰을 비교하세요. 토큰 공간에서 '반전'이 보이나요? 무엇으로 그렇게 판단하나요?

**C. 공식 파일 주석의 출처 (정답 없음)** — YAML 주석의 `436 (64+12+116+116+116+12)`은 어떤 구성이었을까요? 29관절이 아니라면 몇 관절일까요? 공식 저장소에서 근거를 찾아보세요.

In [ ]:
# 🚀 A. ✍️ Mode-0 Encoder 입력
def build_encoder_obs_mode0(motion, frame, robot_xyzw, alignment):
    raise NotImplementedError

# from my_work.day1 import *
# check.encoder_obs_mode0(lab, build_encoder_obs_mode0)